<a href="https://colab.research.google.com/github/simichakravarty16/walmart-hurricane-project/blob/main/notebooks/team_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Walmart Harricane Midterm Project
Group Members: Xiangyu Shi and Simi Chakravarty

## Introduction

Description of the project and the dataset we will use

### 1. Import Libraries

In [ ]:
import pandas as pd
import os
import sqlite3
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### 2. Define Decision log



In [ ]:
DECISIONS = []

def log(dataset, step, decision, rows_affected):
    DECISIONS.append({'dataset': dataset, 'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{dataset}] [{step}] {decision} ({rows_affected} row(s))')

### 3. Data Loading and Inspection

In [ ]:
# Set data directory
DATA_DIR = '/content'

# Define data loading function
def load_data(filename):
  return pd.read_csv(os.path.join(DATA_DIR, filename))

In [ ]:
# Load store locations dataset
stores = load_data('store_locations.csv')
stores

,store_id,store_name,city,state,zip_code,latitude,longitude,region
0,FL-239,Walmart Supercenter Orlando,Orlando,FL,32801,28.5383,-81.3792,Central
1,FL-105,Walmart Supercenter Tampa,Tampa,FL,33602,27.9506,-82.4572,West
2,FL-330,Walmart Supercenter Miami,Miami,FL,33101,25.7617,-80.1918,South
3,FL-412,Walmart Supercenter Jacksonville,Jacksonville,FL,32099,30.3322,-81.6557,North
4,FL-287,Walmart Neighborhood Market Kissimmee,Kissimmee,Florida,34741,28.2920,-81.4076,Central
5,FL-156,Walmart Supercenter Daytona Beach,Daytona Beach,FL,32114,29.2108,-81.0228,East
6,FL-501,Walmart Supercenter Fort Myers,Fort Myers,FL,33901,26.6406,-81.8723,West
7,FL-078,Walmart Supercenter Gainesville,Gainesville,FL,32601,29.6516,-82.3248,North
8,FL-619,Walmart Supercenter Tallahassee,Tallahassee,FL,32301,NaN,NaN,Panhandle
9,FL-445,Walmart Supercenter St. Petersburg,St. Petersburg,Fla.,33701,27.7676,-82.6403,West


In [ ]:
# Count number of rows in the dataset
raw_rows = len(stores)
print('starting with', raw_rows, 'rows')
print()

# take inventory
stores.info()

starting with 14 rows

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   store_id    14 non-null     object 
 1   store_name  14 non-null     object 
 2   city        14 non-null     object 
 3   state       14 non-null     object 
 4   zip_code    14 non-null     int64  
 5   latitude    13 non-null     float64
 6   longitude   13 non-null     float64
 7   region      14 non-null     object 
dtypes: float64(2), int64(1), object(5)
memory usage: 1.0+ KB


In [ ]:
stores.describe()

,zip_code,latitude,longitude
count,14.000000,13.000000,13.000000
mean,33200.428571,28.418192,-82.069631
std,893.995759,1.319349,1.676108
min,32099.000000,25.761700,-87.216900
25%,32526.000000,27.950600,-82.324800
50%,32951.000000,28.292000,-81.655700
75%,33776.000000,29.210800,-81.379200
max,34741.000000,30.421300,-80.191800


In [ ]:
print('shape:', stores.shape)
print()
print('dtypes:', stores.dtypes)
print()
print('nulls:', stores.isnull().sum())
print()
print('duplicates:', stores.duplicated().sum())

shape: (14, 8)

dtypes: store_id       object
store_name     object
city           object
state          object
zip_code        int64
latitude      float64
longitude     float64
region         object
dtype: object

nulls: store_id      0
store_name    0
city          0
state         0
zip_code      0
latitude      1
longitude     1
region        0
dtype: int64

duplicates: 0


Problems:
1. The `store_id` column does not consistently follow the `FL-#` format.
2. Not all values in the `state` column are written as `FL`.  
3. The `zip_code` column is stored as `int` instead of `str`.
4. There is one null in each of the `latitude` and `longitude` column for store `FL-619`.
5. There is a leading space before `Panhandle` in the `region` column for `FL-619`.



In [ ]:
# Load distribution centers dataset
dc = load_data('distribution_centers.csv')
dc

,dc_id,dc_name,city,state,avg_daily_shipments,surge_capacity,distance_to_orlando_mi,cost_per_shipment
0,DC-JAX,Jacksonville Distribution Center,Jacksonville,FL,500,1000,141,245.0
1,DC-ATL,Atlanta Distribution Center,Atlanta,GA,400,700,438,412.5
2,DC-DAL,Dallas Distribution Center,Dallas,TX,300,600,1108,687.0
3,DC-MIA,Miami Distribution Center,Miami,FL,350,~750,235,298.0
4,DC-BHM,Birmingham Distribution Center,Birmingham,AL,250,500,503,NaN


In [ ]:
# Count number of rows in the dataset
raw_rows = len(dc)
print('starting with', raw_rows, 'rows')
print()

# take inventory
dc.info()

starting with 5 rows

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   dc_id                   5 non-null      object 
 1   dc_name                 5 non-null      object 
 2   city                    5 non-null      object 
 3   state                   5 non-null      object 
 4   avg_daily_shipments     5 non-null      int64  
 5   surge_capacity          5 non-null      object 
 6   distance_to_orlando_mi  5 non-null      int64  
 7   cost_per_shipment       4 non-null      float64
dtypes: float64(1), int64(2), object(5)
memory usage: 452.0+ bytes


In [ ]:
dc.describe()

,avg_daily_shipments,distance_to_orlando_mi,cost_per_shipment
count,5.00000,5.000000,4.000000
mean,360.00000,485.000000,410.625000
std,96.17692,377.987434,197.064023
min,250.00000,141.000000,245.000000
25%,300.00000,235.000000,284.750000
50%,350.00000,438.000000,355.250000
75%,400.00000,503.000000,481.125000
max,500.00000,1108.000000,687.000000


In [ ]:
print('shape:', dc.shape)
print()
print('dtypes:', dc.dtypes)
print()
print('nulls:', dc.isnull().sum())
print()
print('duplicates:', dc.duplicated().sum())

shape: (5, 8)

dtypes: dc_id                      object
dc_name                    object
city                       object
state                      object
avg_daily_shipments         int64
surge_capacity             object
distance_to_orlando_mi      int64
cost_per_shipment         float64
dtype: object

nulls: dc_id                     0
dc_name                   0
city                      0
state                     0
avg_daily_shipments       0
surge_capacity            0
distance_to_orlando_mi    0
cost_per_shipment         1
dtype: int64

duplicates: 0


Problems:
1.   The `surge_capacity` column is stored as `object` instead of `int`.
2.   There is one null in the `cost_per_shipment` column.





In [ ]:
def clean_distribution_centers(df):
    out = df.copy()
    # Clean surge_capacity
    affected = out['surge_capacity'].str.contains('~').sum()
    # Remove ~ and convert surge_capacity to integer
    out['surge_capacity'] = out['surge_capacity'].str.replace('~', '').astype(int)
    # log decision
    log('distribution_centers', 'clean surge_capacity', 'Removed ~ and converted to integer', affected)


,dc_id,dc_name,city,state,avg_daily_shipments,surge_capacity,distance_to_orlando_mi,cost_per_shipment
0,DC-JAX,Jacksonville Distribution Center,Jacksonville,FL,500,1000,141,245.0
1,DC-ATL,Atlanta Distribution Center,Atlanta,GA,400,700,438,412.5
2,DC-DAL,Dallas Distribution Center,Dallas,TX,300,600,1108,687.0
3,DC-MIA,Miami Distribution Center,Miami,FL,350,750,235,298.0
4,DC-BHM,Birmingham Distribution Center,Birmingham,AL,250,500,503,NaN


[distribution_centers] [clean surge_capacity] Removed ~ and converted to integer (1 row(s))


Solutions:

1.  Remove the `~` symbol for one value in the `surge_capacity` column and covert the column to integer type.
2.  Keep the distribution center row with the missing `cost_per_shipment` value to avoid excluding one of the distribution centers from the dataset.

